In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score 
from sklearn.ensemble import GradientBoostingClassifier
from pathlib import Path
from sklearn.model_selection import train_test_split

In [3]:
DATA_DIR = Path(r"C:\Users\DELL\Datasets\Will_buy_EV_dataset")

df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

In [4]:
TARGET = 'Will_Buy_EV'
CAT = ['Gender','City_Type', 'Current_Car_Type']
CAT_2 = 'Range_Anxiety_Level'
BOOL = ['Home_Charging_Possible','Subsidy_Available']
NUM = ['Age', 'Annual_Income_USD', 'Daily_Commute_km',
       'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
       'Charging_Stations_Near_Work', 'Environmental_Concern_Level']

In [6]:
def encode(d):
    d = d.copy()
    d[CAT_2] = d[CAT_2].map({'Low': 0, 'Medium': 1, 'High': 2})
    for bol in BOOL:
        d[bol] = d[bol].map({'No': 0, 'Yes': 1})
    return pd.get_dummies(d, columns=CAT, dtype=int)


def preprocess(df):
    df = df.drop(columns=['id']).copy()
    if not pd.api.types.is_numeric_dtype(df[TARGET]):    # 'Yes'/'No' strings -> 1/0
        df[TARGET] = df[TARGET].map({'No': 0, 'Yes': 1})

    train, valid = train_test_split(df, test_size=0.2, random_state=42, stratify=df[TARGET])
    train, valid = encode(train), encode(valid)
    valid = valid.reindex(columns=train.columns, fill_value=0)   # align one-hot columns

    sc = StandardScaler()
    train[NUM] = sc.fit_transform(train[NUM])    # fit on train only
    valid[NUM] = sc.transform(valid[NUM])        # reuse train statistics

    assert not train.isna().any().any() and not valid.isna().any().any(), "unmapped value -> NaN"

    x_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
    x_valid, y_valid = valid.drop(columns=[TARGET]), valid[TARGET]
    return x_train, y_train, x_valid, y_valid, sc

In [7]:
x_train, y_train, x_valid, y_valid, sc = preprocess(df)
print(x_train.shape, x_valid.shape)
print(y_train.mean())   # share of buyers, should match in both splits

(534932, 20) (133733, 20)
0.17464462772838416


In [8]:
def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        s = model.predict_proba(X)
        return s[:, 1] if s.ndim == 2 else s
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    print("WARNING: using hard labels, AUC will be misleading")
    return model.predict(X)


def get_roc(model, X, y):
    auc = roc_auc_score(y, get_scores(model, X))
    print("ROC_AUC_SCORE :", auc)
    return auc

In [9]:
x_small, _, y_small, _ = train_test_split(
    x_train, y_train, train_size=0.25, stratify=y_train, random_state=42)

In [11]:
quick = GradientBoostingClassifier(n_estimators=50)
%time quick.fit(x_small, y_small)

CPU times: total: 10.5 s
Wall time: 10.6 s


GradientBoostingClassifier(n_estimators=50)

In [12]:
boost = GradientBoostingClassifier()
model = boost.fit(x_train, y_train)

In [13]:
get_roc(model, x_valid, y_valid)

ROC_AUC_SCORE : 0.9397916314861429


np.float64(0.9397916314861429)

In [17]:
test_ids = test_df['id']
x_test = encode(test_df.drop(columns=['id'])).reindex(columns=x_train.columns, fill_value=0)
x_test[NUM] = sc.transform(x_test[NUM])

submission = pd.DataFrame({'id':test_ids, 'TARGET':get_scores(model, x_test)})

In [18]:
submission.head(10)

,id,TARGET
0,668665,0.011812
1,668666,0.011299
2,668667,0.008255
3,668668,0.003745
4,668669,0.022538
5,668670,0.006962
6,668671,0.399437
7,668672,0.001902
8,668673,0.678651
9,668674,0.009864
